# Block 1 lecture — Online Retail II

The lecture's queries, as Eduardo ran them: **for review after class. Nothing here is typed in class.** The cells are numbered as the slides call them. The file is `data/raw/online_retail.parquet`; the lab's file is never opened here.

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 400)   # up to 400 rows in full; a longer result prints head and tail with "..." between: count it, then census it by group

# Anchor to the project folder (DS1, Block 1), then stand there: every path below is from the project folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Working in:", Path.cwd())   # must print this project's folder

con = duckdb.connect()             # an in-memory database; it reads the files in data/raw/ directly

### Cell 1 — `DESCRIBE`: what did DuckDB decide?

In [ ]:
con.sql("""
    DESCRIBE SELECT * FROM 'data/raw/online_retail.parquet'
""").df()

### Cell 2 — `SUMMARIZE`: what is missing, what is impossible?

In [ ]:
con.sql("""
    SUMMARIZE SELECT * FROM 'data/raw/online_retail.parquet'
""").df()

### Cell 3 — the key test. Equal, or not?

In [ ]:
con.sql("""
    SELECT COUNT(*)                            AS n_rows,
           COUNT(DISTINCT (Invoice, StockCode)) AS n_keys
    FROM 'data/raw/online_retail.parquet'
""").df()

### Cell 4 — the census of `Country`: every value, read to the end

In [ ]:
con.sql("""
    SELECT Country, COUNT(*) AS n
    FROM 'data/raw/online_retail.parquet'
    GROUP BY Country
    ORDER BY n DESC
""").df()

### Cell 5 — `SELECT` · `FROM` · `WHERE` · `ORDER BY` · `LIMIT`: the ten largest quantities ordered from Japan

The prediction was the quotes. The `try` prints DuckDB's refusal of `"Japan"` in double quotes (a column name to SQL) and keeps *Run All* going; then the query as written, with `'Japan'` in single quotes (text).

In [ ]:
try:
    con.sql("""
    SELECT Invoice, Description, Quantity, Price
    FROM 'data/raw/online_retail.parquet'
    WHERE Country = "Japan"
    ORDER BY Quantity DESC
    LIMIT 10
""").df()
except duckdb.Error as e:
    print('With "Japan" in double quotes, DuckDB refused:', str(e).splitlines()[0])

con.sql("""
    SELECT Invoice, Description, Quantity, Price
    FROM 'data/raw/online_retail.parquet'
    WHERE Country = 'Japan'
    ORDER BY Quantity DESC
    LIMIT 10
""").df()

### Cell 6 — rows that are not the thing: stock codes that are not products

In [ ]:
con.sql("""
    SELECT DISTINCT StockCode, Description
    FROM 'data/raw/online_retail.parquet'
    WHERE StockCode IN ('POST', 'DOT', 'C2', 'BANK CHARGES')
    ORDER BY StockCode
""").df()

### Cell 7 — text that looks like a number: `CAST` refuses

`StockCode` is `VARCHAR`: some codes end in a letter (`85123A`). `CAST` stops at the first value it cannot convert and names it; which one it meets first can change from run to run. The `try` only keeps a *Run All* going past the refusal; in class the error is shown as it is.

In [ ]:
try:
    con.sql("""
        SELECT CAST(StockCode AS INTEGER) AS code
        FROM 'data/raw/online_retail.parquet'
    """).df()
except duckdb.Error as e:
    print("DuckDB refused:", str(e).splitlines()[0])

### Cell 8 — `TRY_CAST`, and what it swallowed

No error. The counts are the only evidence: `converted` against `lines`. The second table is the codes it turned into `NULL`, the most frequent first.

In [ ]:
print(con.sql("""
    SELECT COUNT(*)                                       AS lines,
           COUNT(TRY_CAST(StockCode AS INTEGER))          AS converted,
           COUNT(DISTINCT StockCode)                      AS codes,
           COUNT(DISTINCT TRY_CAST(StockCode AS INTEGER)) AS codes_converted
    FROM 'data/raw/online_retail.parquet'
""").df())

con.sql("""
    SELECT StockCode, COUNT(*) AS lines
    FROM 'data/raw/online_retail.parquet'
    WHERE TRY_CAST(StockCode AS INTEGER) IS NULL
    GROUP BY StockCode
    ORDER BY lines DESC, StockCode
    LIMIT 10
""").df()

### Cell 9 — `NULL` in a `WHERE`: three filters, three counts. Do the first two add up to 525,461?

In [ ]:
for cond in ["Description = 'Manual'", "Description != 'Manual'", "Description IS NULL"]:
    n = con.sql(f"SELECT COUNT(*) FROM 'data/raw/online_retail.parquet' WHERE {cond}").fetchone()[0]
    print(f"{cond:28} {n:>9,}")

### Cell 10 — `= NULL` is never true

In [ ]:
con.sql("""
    SELECT COUNT(*) FROM 'data/raw/online_retail.parquet' WHERE Description = NULL
""").df()

### Cell 11 — `NULL` in a count

In [ ]:
con.sql("""
    SELECT COUNT(*), COUNT("Customer ID"), COUNT(DISTINCT "Customer ID")
    FROM 'data/raw/online_retail.parquet'
""").df()

### Cell 12 — `NULL` in arithmetic, a sum, and an average

Three real cells, unchanged, of Eurostat's municipal waste file (`env_wasmun`, the row `A,GEN,KG_HAB,AT`: Austria, kilograms per inhabitant): 2022 `803`, 2023 `782`, 2024 `:`. Eurostat writes `:` for *not available*; here it is `NULL`. The shop's file has no measure that is ever missing.

In [ ]:
con.sql("""
    SELECT 803 + 782 + NULL AS with_plus,
           SUM(kg)          AS with_sum,
           AVG(kg)          AS average
    FROM (VALUES (2022, 803), (2023, 782), (2024, NULL)) AS t(yr, kg)
""").df()

---

## What to remember from Block 1

1. **A table is a claim: one row is one ___.** That is its *grain*. The *key* is the grain written as columns:
   no two rows share it, and it is never missing. Both are claims you test, not facts you assume.
2. **The inspection reflex, on every table, before any number:** `DESCRIBE` (types) · `SUMMARIZE` (what is missing,
   what is impossible) · the key test `COUNT(*)` against `COUNT(DISTINCT (key))` · the census, read to the end.
3. Here, `(Invoice, StockCode)` is **not** a key: 525,461 rows, 512,126 keys. Some products appear twice on an
   invoice, and 6,865 lines are exact copies. Whether a copy is a second sale is a decision; write it down.
4. **Rows that are not the thing** sit in the table like any other row: postage, bank charges, a country called
   *Unspecified*. `SUM` and `COUNT(*)` add them without a word. The census finds them.
5. **Types:** one code with a letter in it (`85123A`) makes `StockCode` text, and `CAST` refuses to make it a number:
   an honest error. `TRY_CAST` turns what it cannot convert into `NULL`, silently: 80,112 of 525,461 lines lost their
   code, 1,675 of 4,632 codes. Count what it swallowed.
6. **NULL:** `= NULL` is never true — use `IS NULL`. `!=` drops the NULL rows (854 + 521,679 ≠ 525,461).
   `COUNT(col)` skips NULL; `SUM` and `AVG` skip NULL; **`+` with a NULL is NULL.** Plus poisons; SUM skips:
   Austria's `803 + 782 + NULL` is `NULL`, its `SUM` is 1,585, its `AVG` is 792.5 (not 528.33).
7. **Units live in the documentation**, never in the column. Read `DATA.md` before the first sum.

## Common mistakes

- **Summing before asking what one row is.** A total over a table that also holds postage lines, copies, or summary
  rows is a number, not an answer. Run the census first.
- **Reading the top of the census and stopping.** The rows that break a report (`Unspecified`, `EIRE`, a space
  before `Bank Charges`) are never in the top five.
- **Writing `= NULL`.** It is never true, so the query returns nothing and no error. Use `IS NULL`.
- **Using `!=` to mean "everything else".** It also drops every row where the column is `NULL`
  (521,679 is not 525,461 − 854). Use `IS DISTINCT FROM`, or add `OR col IS NULL`.
- **Trusting a `TRY_CAST` result.** It turns what it cannot read into `NULL` and carries on: here it would have
  dropped `85123A`, a real product, and kept no trace. Count what it swallowed.
- **Adding a missing value with `+`.** `803 + 782 + NULL` is `NULL`: the whole result disappears. `SUM` skips `NULL`.
- **Double quotes around text.** `"Japan"` is a column name to SQL; the value is `'Japan'`.